# finrag: full experiment run

Thin runner for the complete pipeline. Each cell calls the same CLI entry
points documented in the README; all logic lives in the package and the
experiment scripts, so this notebook cannot drift from the reproducible path.

Requirements: a CUDA GPU, uv, and Ollama with the five models pulled
(scripts/setup_pod.sh does all of this on a fresh pod).

## 1. Environment check

In [ ]:
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available() and torch.cuda.get_device_name(0))"
!uv run pytest

## 2. Corpus: download and extract

In [ ]:
!uv run finrag download

In [ ]:
!uv run finrag extract

## 3. Retrieval grid (cached and resumable)

In [ ]:
!uv run python experiments/run_grid.py --sweep all --device cuda

In [ ]:
!uv run python experiments/analyze_retrieval.py

## 4. Generation sweep (five models) and ablations

In [ ]:
!uv run python experiments/run_generation.py --all -k 10

In [ ]:
!uv run python experiments/run_generation.py --gen-model qwen2.5:14b-instruct-q4_K_M --context-mode oracle
!uv run python experiments/run_generation.py --gen-model qwen2.5:7b-instruct --context-mode oracle
!uv run python experiments/run_generation.py --gen-model qwen2.5:14b-instruct-q4_K_M --context-mode none --judge none
!uv run python experiments/run_generation.py --gen-model qwen2.5:7b-instruct --context-mode none --judge none

In [ ]:
!uv run python experiments/analyze_generation.py

## 5. Judge validation (secondary judge: gpt-5.5 via Codex CLI)

In [ ]:
!uv run python experiments/judge_agreement.py --sample 60

## 6. Inspect the outputs

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display
tables = Path('results/tables')
for name in ['retrieval_results.md', 'generation_results.md', 'error_analysis.md', 'retrieval_vs_generation.md']:
    p = tables / name
    if p.exists():
        display(Markdown(p.read_text()))
for p in sorted(Path('results/figures').glob('*.png')):
    display(Image(str(p)))